In [0]:
%run ../00-common/01-env-variables

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)
drivers_required_df = drivers_df.drop("url")
display(drivers_required_df)

In [0]:
renamed_driver_dff = drivers_required_df.withColumnsRenamed(
    {
        'driverId': 'driver_id',
        'dateOfBirth' : 'date_of_birth'
    }
)

In [0]:
distinct_drivers_record = renamed_driver_dff.dropDuplicates(['driver_id'])

In [0]:
from pyspark.sql import functions as F
title_case_drivers_df = distinct_drivers_record.withColumn('nationality', F.initcap(F.col('nationality')))

In [0]:
display(title_case_drivers_df)

In [0]:
from pyspark.sql import functions as F
concatinated_drivers_df = title_case_drivers_df.withColumn('name' , F.initcap(F.concat_ws(" " , F.col('name.givenName') , F.col('name.familyName')) ))

display(concatinated_drivers_df)

In [0]:
(concatinated_drivers_df.write
.mode('overwrite')
.option("overwriteSchema", "true")
.saveAsTable(silver_table)
)